In [1]:
from pyspark.sql import SparkSession

In [3]:
spark = (
    SparkSession
    .builder
    .appName("SparkLabNotebook")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/09 21:42:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark.version

'4.2.0'

In [5]:
data = [
    ("Alice", 25, "Chicago"),
    ("Bob", 32, "New York"),
    ("Charlie", 28, "Chicago"),
    ("David", 41, "Boston"),
    ("Eve", 35, "Chicago"),
]

In [6]:
df = spark.createDataFrame(
    data,
    ["name", "age", "city"]
)

In [7]:
df.show()

+-------+---+--------+
|   name|age|    city|
+-------+---+--------+
|  Alice| 25| Chicago|
|    Bob| 32|New York|
|Charlie| 28| Chicago|
|  David| 41|  Boston|
|    Eve| 35| Chicago|
+-------+---+--------+



In [8]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)



In [11]:
df.select("name", "age").show()

+-------+---+
|   name|age|
+-------+---+
|  Alice| 25|
|    Bob| 32|
|Charlie| 28|
|  David| 41|
|    Eve| 35|
+-------+---+



In [12]:
df.filter(df.age > 30).show()

+-----+---+--------+
| name|age|    city|
+-----+---+--------+
|  Bob| 32|New York|
|David| 41|  Boston|
|  Eve| 35| Chicago|
+-----+---+--------+



In [19]:
df.age

Column<'age'>

In [13]:
df.filter("age > 30").show()

+-----+---+--------+
| name|age|    city|
+-----+---+--------+
|  Bob| 32|New York|
|David| 41|  Boston|
|  Eve| 35| Chicago|
+-----+---+--------+



In [14]:
df.orderBy("age").show()

+-------+---+--------+
|   name|age|    city|
+-------+---+--------+
|  Alice| 25| Chicago|
|Charlie| 28| Chicago|
|    Bob| 32|New York|
|    Eve| 35| Chicago|
|  David| 41|  Boston|
+-------+---+--------+



In [15]:
from pyspark.sql.functions import desc

df.orderBy(desc("age")).show()

+-------+---+--------+
|   name|age|    city|
+-------+---+--------+
|  David| 41|  Boston|
|    Eve| 35| Chicago|
|    Bob| 32|New York|
|Charlie| 28| Chicago|
|  Alice| 25| Chicago|
+-------+---+--------+



In [16]:
from pyspark.sql import functions as F

In [17]:
older = df.withColumn(
    "age_next_year",
    F.col("age") + 1
)
older.show()

+-------+---+--------+-------------+
|   name|age|    city|age_next_year|
+-------+---+--------+-------------+
|  Alice| 25| Chicago|           26|
|    Bob| 32|New York|           33|
|Charlie| 28| Chicago|           29|
|  David| 41|  Boston|           42|
|    Eve| 35| Chicago|           36|
+-------+---+--------+-------------+



In [20]:
df.groupBy("city").count().show()

+--------+-----+
|    city|count|
+--------+-----+
| Chicago|    3|
|New York|    1|
|  Boston|    1|
+--------+-----+



In [21]:
df.groupBy("city").agg(
    F.avg("age").alias("average_age")
).show()

+--------+------------------+
|    city|       average_age|
+--------+------------------+
| Chicago|29.333333333333332|
|New York|              32.0|
|  Boston|              41.0|
+--------+------------------+



# Spark SQL

In [22]:
df.createOrReplaceTempView("people")

In [23]:
spark.sql("""
SELECT *
FROM people
""").show()

+-------+---+--------+
|   name|age|    city|
+-------+---+--------+
|  Alice| 25| Chicago|
|    Bob| 32|New York|
|Charlie| 28| Chicago|
|  David| 41|  Boston|
|    Eve| 35| Chicago|
+-------+---+--------+



In [24]:
spark.sql("""
    SELECT
        city,
        COUNT(*) AS people,
        AVG(age) AS average_age
    FROM people
    GROUP BY city
    ORDER BY people DESC
""").show()

+--------+------+------------------+
|    city|people|       average_age|
+--------+------+------------------+
| Chicago|     3|29.333333333333332|
|New York|     1|              32.0|
|  Boston|     1|              41.0|
+--------+------+------------------+



In [25]:
result = (
    df
    .filter(F.col("age") > 25)
    .groupBy("city")
    .agg(F.avg("age"))
)

In [26]:
result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[city#2], functions=[avg(age#1L)])
   +- Exchange hashpartitioning(city#2, 200), ENSURE_REQUIREMENTS, [plan_id=261]
      +- HashAggregate(keys=[city#2], functions=[partial_avg(age#1L)])
         +- Project [age#1L, city#2]
            +- Filter (isnotnull(age#1L) AND (age#1L > 25))
               +- Scan ExistingRDD[name#0,age#1L,city#2]




In [27]:
result.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [3]: [name#0, age#1L, city#2]
Arguments: [name#0, age#1L, city#2], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [3]: [name#0, age#1L, city#2]
Condition : (isnotnull(age#1L) AND (age#1L > 25))

(3) Project
Output [2]: [age#1L, city#2]
Input [3]: [name#0, age#1L, city#2]

(4) HashAggregate
Input [2]: [age#1L, city#2]
Keys [1]: [city#2]
Functions [1]: [partial_avg(age#1L)]
Aggregate Attributes [2]: [sum#143, count#144L]
Results [3]: [city#2, sum#145, count#146L]

(5) Exchange
Input [3]: [city#2, sum#145, count#146L]
Arguments: hashpartitioning(city#2, 200), ENSURE_REQUIREMENTS, [plan_id=261]

(6) HashAggregate
Input [3]: [city#2, sum#145, count#146L]
Keys [1]: [city#2]
Functi

In [28]:
result.show()

+--------+--------+
|    city|avg(age)|
+--------+--------+
|New York|    32.0|
| Chicago|    31.5|
|  Boston|    41.0|
+--------+--------+



In [30]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("people.csv")
)

In [31]:
df.show()

+-------+---+--------+
|   name|age|    city|
+-------+---+--------+
|  Alice| 25| Chicago|
|    Bob| 32|New York|
|Charlie| 28| Chicago|
|  David| 41|  Boston|
|    Eve| 35| Chicago|
|  Frank| 29|  Boston|
|  Grace| 38|New York|
|  Henry| 22| Chicago|
+-------+---+--------+



In [32]:
df.write.mode("overwrite").parquet("people.parquet")

In [33]:
parquet_df = spark.read.parquet("people.parquet")
parquet_df.show()

+-------+---+--------+
|   name|age|    city|
+-------+---+--------+
|  Alice| 25| Chicago|
|    Bob| 32|New York|
|Charlie| 28| Chicago|
|  David| 41|  Boston|
|    Eve| 35| Chicago|
|  Frank| 29|  Boston|
|  Grace| 38|New York|
|  Henry| 22| Chicago|
+-------+---+--------+



In [34]:
!ls -lah people.parquet

total 24
-rw-r--r--@  1 christague  staff     0B Aug  9 21:55 _SUCCESS
drwxr-xr-x@  6 christague  staff   192B Aug  9 21:55 .
-rw-r--r--@  1 christague  staff     8B Aug  9 21:55 ._SUCCESS.crc
drwxr-xr-x@ 15 christague  staff   480B Aug  9 21:56 ..
-rw-r--r--@  1 christague  staff    20B Aug  9 21:55 .part-00000-52ee6f0a-01a2-43b9-9e1d-17d50b2baa29-c000.snappy.parquet.crc
-rw-r--r--@  1 christague  staff   1.1K Aug  9 21:55 part-00000-52ee6f0a-01a2-43b9-9e1d-17d50b2baa29-c000.snappy.parquet


In [36]:
df.rdd.getNumPartitions()

1